Data Ingestion Pipeline

In [2]:
from langchain_core.documents import Document

In [3]:
from langchain_community.document_loaders import TextLoader,DirectoryLoader

C:\Users\Tashu\AppData\Local\Temp\ipykernel_33768\4160123187.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader,DirectoryLoader


In [4]:
loader = TextLoader(
    "C:/Users/Tashu/Desktop/ragLearning/data/text1.txt",
    encoding="utf-8"
)
document = loader.load() 
print(document)

[Document(metadata={'source': 'C:/Users/Tashu/Desktop/ragLearning/data/text1.txt'}, page_content='LangChain is an open-source software framework designed to help developers build applications powered by large language models (LLMs) like GPT, Claude, and Gemini.\n\nWhat LangChain Does\n\nRaw LLMs have limitations: they lack long-term memory, cannot read private documents out-of-the-box, and cannot natively execute real-world actions or call external APIs. LangChain acts as the connective "plumbing" between LLMs and external systems.\nIt is available in both Python and JavaScript.\n\nCore Components\n\n• Model Wrappers: Standardized interfaces that let you switch or connect to hundreds of different LLMs easily.\n• Prompt Templates: Dynamic structures to format and inject variables (like user names, dates, or specific contexts) into prompts.\n• Chains: Pipelines that link multiple steps together—such as taking user input, running it through a prompt template, sending it to an LLM, and par

For extracting all similar kind of files and above cell if a single file

In [5]:
dir_loader = DirectoryLoader(
    "C:/Users/Tashu/Desktop/ragLearning/data",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs= {'encoding':'utf-8'},
    show_progress=False
)   

In [6]:
documents = dir_loader.load()
print(documents)

[Document(metadata={'source': 'C:\\Users\\Tashu\\Desktop\\ragLearning\\data\\text1.txt'}, page_content='LangChain is an open-source software framework designed to help developers build applications powered by large language models (LLMs) like GPT, Claude, and Gemini.\n\nWhat LangChain Does\n\nRaw LLMs have limitations: they lack long-term memory, cannot read private documents out-of-the-box, and cannot natively execute real-world actions or call external APIs. LangChain acts as the connective "plumbing" between LLMs and external systems.\nIt is available in both Python and JavaScript.\n\nCore Components\n\n• Model Wrappers: Standardized interfaces that let you switch or connect to hundreds of different LLMs easily.\n• Prompt Templates: Dynamic structures to format and inject variables (like user names, dates, or specific contexts) into prompts.\n• Chains: Pipelines that link multiple steps together—such as taking user input, running it through a prompt template, sending it to an LLM, a

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

dir_loader_pdf = DirectoryLoader(
    "C:/Users/Tashu/Desktop/ragLearning/data",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)   

doc1 = dir_loader_pdf.load()
print(doc1)

In [ ]:
from pathlib import Path

def process_pdf(path):
    all_docs = []
    path_dir = Path(path)
    pdf_files = list(path_dir.glob("**/*.pdf"))
    print(len(pdf_files),"are to be loaded")
    for pdf_file in pdf_files:
        print(f"processing {pdf_file.name}")
        try :
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'
            print(f"Loaded {len(documents)} pages")
            all_docs.extend(documents)

        except :
            print("Error X")
    print("All Docs loaded")
    return all_docs

all_pdf_docs = process_pdf("C:/Users/Tashu/Desktop/ragLearning/data")

print(all_pdf_docs)

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_split(documents,chunk_size =1000,chunk_overlap = 200):
    
    text_splitter = RecursiveCharacterTextSplitter(
                                                chunk_size =chunk_size,
                                                chunk_overlap = chunk_overlap,
                                                length_function = len,
                                                separators = ["\n\n","\n"," ",""]
                                                )

    split_docs = text_splitter.split_documents(documents)
    print(f"Splitted {len(documents)} into {len(split_docs)} chunks")
    print(split_docs[0].page_content[:100])
    print(split_docs[0].metadata)

    return split_docs

In [10]:
chunks = chunk_split(all_pdf_docs)
print("Done")


Splitted 73 into 171 chunks
Artificial Intelligence — A Reference Corpus for
Retrieval Testing
Document type: Synthetic referenc
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-23T19:14:54+00:00', 'source': 'C:\\Users\\Tashu\\Desktop\\ragLearning\\data\\AIcorpus.pdf', 'file_path': 'C:\\Users\\Tashu\\Desktop\\ragLearning\\data\\AIcorpus.pdf', 'total_pages': 25, 'format': 'PDF 1.4', 'title': 'Artificial Intelligence - Reference Corpus for RAG Testing', 'author': '(anonymous)', 'subject': 'AI reference corpus with gold question set', 'keywords': '', 'moddate': '2026-10-23T19:14:54+00:00', 'trapped': '', 'modDate': "D:20261023191454+00'00'", 'creationDate': "D:20261023191454+00'00'", 'page': 0, 'source_file': 'AIcorpus.pdf', 'file_type': 'pdf'}
Done


In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from typing import Dict , Tuple , Any , List
from sklearn.metrics.pairwise import cosine_similarity
import uuid

In [12]:
class Embedding_Manager:
    def __init__(self,model_name : str="all-miniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self):
        try:
            print(f"Loading model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded ; dimension of embeddings : {self.model.get_sentence_embedding_dimension}")

        except Exception as e:
            print(f"Error : {e}")
            raise

    def generate_embeddings(self, text : List[str])-> np.ndarray:

        if not self.model:
            raise ValueError("model not present")

        print(f"generating embeddings for {len(text)} texts")
        embeddings = self.model.encode(text,show_progress_bar=True)
        print(f"{len(embeddings)} are generated with shape {embeddings.shape}")
        return embeddings

In [13]:
embedding_manager = Embedding_Manager()
embedding_manager

Loading model : all-miniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6437.68it/s]


Model loaded ; dimension of embeddings : <bound method SentenceTransformer.get_sentence_embedding_dimension of SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)>


In [14]:
import os


class VectorStore :

    def __init__(self , collection_name = "pdf_docs", persist_path = "C:/Users/Tashu/Desktop/ragLearning/data/vectordb" ):
        self.collection_name = collection_name
        self.persist_directory = persist_path
        self.client = None
        self.collection = None
        self.initialize_store()

    def initialize_store(self):
        try:
            # os.mkdirs(self.persist_directory , exist_ok = True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {
                    "Description":"embedding of pdf files"
                }
            )

            print(self.collection_name)
            print(self.collection.count())

        except Exception as e :
            print("Error :", e)
            raise

    def add_documents(self , documents : List[Any] , embeddings : np.ndarray):

        if(len(documents)!=len(embeddings)):
            raise ValueError("ERROR")

        ids = []
        metadatas = []
        doc_text = []
        embedding_list = []

        for i , (doc,emb) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['index_doc'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            doc_text.append(doc.page_content)
            embedding_list.append(emb.tolist())

            try:
                self.collection.add(
                    ids = ids,
                    embeddings = embedding_list,
                    metadatas = metadatas,
                    documents = doc_text
                )

                print("total documents in database :",self.collection.count())

            except Exception as e:
                raise ValueError(f"Not stored in vectorDb : {e}")

In [15]:
vectorStore = VectorStore()
vectorStore

pdf_docs
171


Converting text to embeddings

In [16]:
texts = [doc.page_content for doc in chunks] #making a list of all chunk's content
embeddings = embedding_manager.generate_embeddings(texts)



generating embeddings for 171 texts


Batches: 100%|██████████| 6/6 [00:01<00:00,  3.21it/s]

171 are generated with shape (171, 384)


In [17]:
#Storing in vectorDb

vectorStore.add_documents(chunks,embeddings)

total documents in database : 172
total documents in database : 173
total documents in database : 174
total documents in database : 175
total documents in database : 176
total documents in database : 177
total documents in database : 178
total documents in database : 179
total documents in database : 180
total documents in database : 181
total documents in database : 182
total documents in database : 183
total documents in database : 184
total documents in database : 185
total documents in database : 186
total documents in database : 187
total documents in database : 188
total documents in database : 189
total documents in database : 190
total documents in database : 191
total documents in database : 192
total documents in database : 193
total documents in database : 194
total documents in database : 195
total documents in database : 196
total documents in database : 197
total documents in database : 198
total documents in database : 199
total documents in database : 200
total document

RAG retrieval pipeline

In [18]:
class Ragretriever :

    def __init__(self , vector_store : VectorStore , embedding_manager : Embedding_Manager):

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager


    def retrieve(self , query : str , top_k : int = 5 , score_threshold : float = 0.0):

        print(f"retrieving docs for query : {query}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0] :
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i , (doc_id , document , metadata , distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    similarity_score = 1-distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id' : doc_id,
                            'content' : document,
                            'metadata' : metadata,
                            'similarity_score' : similarity_score,
                            'distance' : distance,
                            'rank' : i+1
                        })

                    print(f"retrieved {len(retrieved_docs)} docs for query")

            else :
                print("No docs found")

            return retrieved_docs

        except:
            print("Error")
            return []


rag_retriever = Ragretriever(vectorStore,embedding_manager)



In [19]:
rag_retriever.retrieve("Artificial Intelligence — A Reference Corpus for Retrieval Testing")

retrieving docs for query : Artificial Intelligence — A Reference Corpus for Retrieval Testing
generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 86.06it/s]

1 are generated with shape (1, 384)
retrieved 1 docs for query
retrieved 2 docs for query
retrieved 3 docs for query
retrieved 4 docs for query
retrieved 4 docs for query


[{'id': 'doc_fd0bc444_0',
  'content': "Artificial Intelligence — A Reference Corpus for\nRetrieval Testing\nDocument type: Synthetic reference corpus, built for testing retrieval-augmented\ngeneration (RAG) pipelines\nScope: Foundations, history, methods, architectures, evaluation, infrastructure, ethics,\ngovernance and applications of artificial intelligence\nStructure: 48 numbered sections (A01–A48), each self-contained, followed by a gold\nquestion set (G01–G30) for scoring retrieval\nChunking note: Each ## section is written to stand alone. Splitting on ## headings\nproduces chunks of roughly 700–1,800 characters that require no surrounding context.\nSection IDs are stable, so they can be used as ground-truth labels when measuring\nretrieval precision and recall.\nContent note: Facts here are deliberately specific — dates, named systems, numeric\nthresholds, formulas — so that retrieval failures are visible rather than maskable by a\nmodel's own prior knowledge. Several sections 

In [20]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
load_dotenv()

groqApiKey = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key = groqApiKey , model_name = "openai/gpt-oss-20b" , temperature = 0.1 , max_tokens = 1024)


In [21]:
def rag(query,llm ,retriever,top_k=3):
    results = retriever.retrieve(query,top_k)
    context = "\n\n".join([doc['content'] for doc in results ]) if results else ""
    if not context :
        print("No relevant doc found for the query")

    prompt = f"""I want an concise answer related to query: {query} and use 
        context : {context} and make sure it is to the point.
        Answer:"""

    response = llm.invoke(prompt.format(context = context,query = query))
    return response.content


In [22]:
q = "What is Artificial Intelligence and machine learning ?"

In [23]:
answer = rag(q,llm,rag_retriever)
print(answer)

retrieving docs for query : What is Artificial Intelligence and machine learning ?
generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 125.02it/s]

1 are generated with shape (1, 384)
retrieved 1 docs for query
retrieved 2 docs for query
retrieved 3 docs for query


**Artificial Intelligence (AI)** is the field that designs systems capable of performing tasks normally associated with human cognition—perception, reasoning, learning, planning, language understanding, and acting in an environment.  Russell and Norvig frame AI along two axes:

| Axis | Human‑like | Rational |
|------|------------|----------|
| **Thinking** | Systems that *think like humans* (imitate human cognition) | Systems that *think rationally* (optimize a formal objective) |
| **Acting** | Systems that *act like humans* (behave in human‑like ways) | Systems that *act rationally* (behave to maximize a measurable goal) |

They argue that the **rational‑agent** view—acting rationally—is the most useful engineering target because it is measurable and does not require copying human cognition.

**Machine Learning (ML)** is a subset of AI that focuses on algorithms that learn from data.  Deep learning is a further subset of ML that uses neural networks with many layers.  ML is not a sy

In [24]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:120] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    print(confidence,sources)
    prompt = f"""Use the following context to answer the question concisely.
Context:
{context}

Question: {query}

Answer:"""
    response = llm.invoke([prompt.format(context = context,query = query)])

    return response.content

In [25]:
ans = rag_advanced(q,rag_retriever,llm)
print(ans)

retrieving docs for query : What is Artificial Intelligence and machine learning ?
generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 125.02it/s]

1 are generated with shape (1, 384)
retrieved 1 docs for query
retrieved 2 docs for query
retrieved 3 docs for query
retrieved 4 docs for query
retrieved 4 docs for query
0.289203941822052 [{'source': 'AIcorpus.pdf', 'page': 0, 'score': 0.289203941822052, 'preview': 'two axes: thinking versus acting, and human-like versus rational. This produces four\ncategories — systems that think lik...'}, {'source': 'AIcorpus.pdf', 'page': 0, 'score': 0.289203941822052, 'preview': 'two axes: thinking versus acting, and human-like versus rational. This produces four\ncategories — systems that think lik...'}, {'source': 'AIcorpus.pdf', 'page': 0, 'score': 0.267306387424469, 'preview': "thresholds, formulas — so that retrieval failures are visible rather than maskable by a\nmodel's own prior knowledge. Sev..."}, {'source': 'AIcorpus.pdf', 'page': 0, 'score': 0.267306387424469, 'preview': "thresholds, formulas — so that retrieval failures are visible rather than maskable by a\nmodel's own prior knowled

**Artificial Intelligence (AI)** is the field that builds systems capable of performing tasks normally associated with human cognition—perceiving, reasoning, learning, planning, understanding language, and acting in an environment.  Russell and Norvig’s framework classifies AI along two axes: *thinking vs. acting* and *human‑like vs. rational*, yielding four categories (think like humans, think rationally, act like humans, act rationally).  The most engineering‑useful target is the *rational‑agent* view: acting rationally, which is measurable and does not require mimicking human cognition.

**Machine Learning (ML)** is a subset of AI that focuses on algorithms that learn patterns from data.  Deep learning is a further subset of ML that uses neural networks with many layers.  ML is not a synonym for AI; rather, it is one of the techniques used within the broader AI field.


In [26]:
import json
import math
import re
from collections import Counter
from sentence_transformers import CrossEncoder

all_data = vectorStore.collection.get(include=['documents', 'metadatas'])
all_documents = [
    {'id': doc_id, 'content': text, 'metadata': metadata or {}}
    for doc_id, text, metadata in zip(all_data['ids'], all_data['documents'], all_data['metadatas'])
]
all_tokens = [re.findall(r'\b\w+\b', doc['content'].lower()) for doc in all_documents]
doc_frequency = Counter(term for tokens in all_tokens for term in set(tokens))
average_length = sum(map(len, all_tokens)) / max(len(all_tokens), 1)
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def bm25_retrieve(query, top_k=12, k1=1.5, b=0.75):
    query_tokens = re.findall(r'\b\w+\b', query.lower())
    scores = []
    for tokens in all_tokens:
        counts = Counter(tokens)
        score = sum(
            math.log(1 + (len(all_documents) - doc_frequency[term] + 0.5) / (doc_frequency[term] + 0.5))
            * (counts[term] * (k1 + 1) / (counts[term] + k1 * (1 - b + b * len(tokens) / average_length)))
            for term in query_tokens if term in doc_frequency
        )
        scores.append(score)
    return [all_documents[index] for index, score in sorted(enumerate(scores), key=lambda item: item[1], reverse=True)[:top_k] if score > 0]

def advanced_rag(query, candidate_k=12, final_k=5):
    dense_documents = rag_retriever.retrieve(query, top_k=candidate_k, score_threshold=-float('inf'))
    dense_documents = [{'id': doc['id'], 'content': doc['content'], 'metadata': doc['metadata']} for doc in dense_documents]
    fused = {}
    for documents in [dense_documents, bm25_retrieve(query, candidate_k)]:
        for rank, document in enumerate(documents, 1):
            fused.setdefault(document['id'], {'document': document, 'score': 0})['score'] += 1 / (60 + rank)
    candidates = [item['document'] for item in sorted(fused.values(), key=lambda item: item['score'], reverse=True)]
    reranked = sorted(zip(candidates, reranker.predict([(query, doc['content']) for doc in candidates])), key=lambda item: float(item[1]), reverse=True)[:final_k]
    passages = '\n\n'.join(f'[{index}] {doc["content"]}' for index, (doc, _) in enumerate(reranked))
    grader = llm.invoke(f'Select passages useful for this question. Return only JSON like {{"relevant_ids": [0]}}.\nQuestion: {query}\nPassages:\n{passages}').content
    try:
        ids = json.loads(re.search(r'\{.*\}', grader, re.DOTALL).group())['relevant_ids']
    except (AttributeError, json.JSONDecodeError, KeyError):
        ids = list(range(len(reranked)))
    context = '\n\n---\n\n'.join(f'Source {index + 1}: {reranked[index][0]["content"]}' for index in ids if isinstance(index, int) and 0 <= index < len(reranked))
    if not context:
        return "I don't know based on the provided PDFs."
    prompt = f'Answer only from this context. If absent, say: I don\'t know based on the provided PDFs. Cite [Source n].\n\n{context}\n\nQuestion: {query}\nAnswer:'
    return llm.invoke(prompt).content

print(advanced_rag(q))


c:\Users\Tashu\Desktop\ragLearning\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tashu\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Tashu\Desktop\ragLearning\.venv\Lib\site-packages\huggingface_hub\file_download.py

retrieving docs for query : What is Artificial Intelligence and machine learning ?
generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 133.86it/s]

1 are generated with shape (1, 384)
retrieved 1 docs for query
retrieved 2 docs for query
retrieved 3 docs for query
retrieved 4 docs for query
retrieved 5 docs for query
retrieved 6 docs for query
retrieved 7 docs for query
retrieved 8 docs for query
retrieved 9 docs for query
retrieved 10 docs for query
retrieved 11 docs for query
retrieved 12 docs for query


Artificial Intelligence (AI) is the field concerned with building systems that perform tasks normally associated with human cognition: perceiving, reasoning, learning, planning, understanding language, and acting in an environment. [Source 1]  

Machine learning is not explicitly defined in the provided PDFs. I don't know based on the provided PDFs. [Source 1]


RAG evaluation on labelled questions (201 for AIcorpus.pdf, 114 for the legal PDFs). The question set is picked automatically from the PDFs loaded: data/eval_questions.json for AIcorpus.pdf (labels are section IDs A01–A48), or data/eval_questions_legal.json for the legal PDFs (labels are file + exact evidence quote). Retrieval metrics: Hit@5, evidence/section recall@5, MRR, Precision@5. Answers are scored by keyword facts, and the unanswerable questions check abstention.

In [ ]:
# ---------------- RAG evaluation harness (~200 questions on AIcorpus.pdf) ----------------
# Uses objects from the cells above: all_pdf_docs, vectorStore, embedding_manager, reranker, llm
import json, math, re, time
from collections import Counter
from pathlib import Path
import numpy as np

def find_data_file(name):
    for base in [Path('../data'), Path('data'), Path('C:/Users/Tashu/Desktop/ragLearning/data')]:
        if (base / name).exists():
            return base / name
    raise FileNotFoundError(f'{name} not found - put it in your data folder')

# Two labelled sets: AIcorpus.pdf (labels = section IDs) and the legal PDFs (labels = file + exact evidence quote).
# Picks the set that matches the PDFs you loaded; override by setting QUESTIONS_FILE yourself.
loaded_files = {d.metadata.get('source_file') for d in all_pdf_docs}
QUESTIONS_FILE = 'eval_questions.json' if 'AIcorpus.pdf' in loaded_files else 'eval_questions_legal.json'
CORPUS_DESCRIPTION = 'a textbook on artificial intelligence' if QUESTIONS_FILE == 'eval_questions.json' \
    else 'a collection of Indian laws and rules on patents, biodiversity, drugs and food'
EVAL_QUESTIONS = json.loads(find_data_file(QUESTIONS_FILE).read_text(encoding='utf-8'))
RESULTS_DIR = find_data_file(QUESTIONS_FILE).parent / 'eval_results'
RESULTS_DIR.mkdir(exist_ok=True)
print(QUESTIONS_FILE, len(EVAL_QUESTIONS), 'questions:', dict(Counter(q['type'] for q in EVAL_QUESTIONS)))

# ---- 1. Label every chunk with the corpus section(s) (A01..A48) it comes from -> ground truth for retrieval metrics
corpus_pages = sorted([d for d in all_pdf_docs if d.metadata.get('source_file') == 'AIcorpus.pdf'],
                      key=lambda d: d.metadata.get('page', 0))
corpus_text = '\n'.join(d.page_content for d in corpus_pages)
section_starts = [(m.start(), m.group(1)) for m in re.finditer(r'^(A\d{2}|G\d{2})(?:–G\d{2})? · ', corpus_text, re.M)]

def sections_of(text, source_file):
    if source_file != 'AIcorpus.pdf' or not corpus_text:
        return [source_file or 'UNKNOWN']   # non-AIcorpus chunks are labelled by file; questions match them by evidence quote
    text = text.strip()
    start = corpus_text.find(text[:200])
    if start < 0:
        start = corpus_text.find(text[:60])
    if start < 0:
        return ['UNKNOWN']
    end = start + len(text)
    current = [sid for pos, sid in section_starts if pos <= start]
    secs = [current[-1]] if current else ['INTRO']
    secs += [sid for pos, sid in section_starts if start < pos < end - 40]   # header must have real text after it
    return list(dict.fromkeys(secs))

# ---- 2. Evaluation index: the unique chunks of the vector store (it holds duplicates from re-running the add cell)
EXCLUDE_GOLD_SET_CHUNKS = True   # the PDF ends with the question set itself (G00-G30); exclude it or the eval is leaked

store = vectorStore.collection.get(include=['documents', 'metadatas'])
eval_docs, seen = [], set()
for doc_id, text, meta in zip(store['ids'], store['documents'], store['metadatas']):
    if text in seen:
        continue
    seen.add(text)
    meta = meta or {}
    secs = sections_of(text, meta.get('source_file'))
    if EXCLUDE_GOLD_SET_CHUNKS and all(s.startswith('G') for s in secs):
        continue
    eval_docs.append({'id': doc_id, 'content': text, 'metadata': meta, 'sections': secs})

eval_emb = embedding_manager.model.encode([d['content'] for d in eval_docs], normalize_embeddings=True, show_progress_bar=False)
eval_tokens = [re.findall(r'\b\w+\b', d['content'].lower()) for d in eval_docs]
eval_counts = [Counter(t) for t in eval_tokens]
eval_df = Counter(term for t in eval_tokens for term in set(t))
eval_avglen = sum(map(len, eval_tokens)) / max(len(eval_tokens), 1)
print(f'{len(eval_docs)} unique chunks indexed;', dict(Counter(s for d in eval_docs for s in d['sections'] if not s.startswith('A'))))

# ---- 3. Retrieval: dense + BM25 -> reciprocal rank fusion -> cross-encoder rerank (same design as advanced_rag)
def dense_search(text, k):
    q = embedding_manager.model.encode([text], normalize_embeddings=True, show_progress_bar=False)[0]
    return list(np.argsort(-(eval_emb @ q))[:k])

def bm25_search(text, k, k1=1.5, b=0.75):
    terms = re.findall(r'\b\w+\b', text.lower())
    n = len(eval_docs)
    scores = []
    for tokens, counts in zip(eval_tokens, eval_counts):
        s = sum(math.log(1 + (n - eval_df[t] + 0.5) / (eval_df[t] + 0.5))
                * counts[t] * (k1 + 1) / (counts[t] + k1 * (1 - b + b * len(tokens) / eval_avglen))
                for t in terms if t in counts)
        scores.append(s)
    order = np.argsort(-np.array(scores))[:k]
    return [i for i in order if scores[i] > 0]

def rrf(ranked_lists, c=60):
    fused = Counter()
    for ranked in ranked_lists:
        for rank, idx in enumerate(ranked, 1):
            fused[idx] += 1 / (c + rank)
    return [idx for idx, _ in fused.most_common()]

def hybrid_retrieve(query, dense_queries=None, sparse_queries=None, candidate_k=12, final_k=5, max_candidates=24):
    """dense_queries / sparse_queries let a query-rewriting step search with different text;
    the cross-encoder always reranks against the ORIGINAL question."""
    lists = [dense_search(t, candidate_k) for t in (dense_queries or [query])]
    lists += [bm25_search(t, candidate_k) for t in (sparse_queries or [query])]
    candidates = rrf(lists)[:max_candidates]
    scores = reranker.predict([(query, eval_docs[i]['content']) for i in candidates], show_progress_bar=False)
    ranked = sorted(zip(candidates, scores), key=lambda x: float(x[1]), reverse=True)[:final_k]
    return [dict(eval_docs[i], rerank_score=float(s)) for i, s in ranked]

# ---- 4. Generation (same prompts as advanced_rag) with retry on Groq rate limits
ABSTAIN = "I don't know based on the provided PDFs."

def call_llm(prompt, retries=6):
    for attempt in range(retries):
        try:
            return llm.invoke(prompt).content
        except Exception as e:
            if attempt == retries - 1 or not re.search(r'429|rate|limit|timeout|503', str(e), re.I):
                raise
            wait = 2 ** attempt * 2
            print(f'  LLM error ({str(e)[:60]}...), retrying in {wait}s')
            time.sleep(wait)

def generate_answer(query, docs, use_grader=True):
    """Returns (answer, number_of_llm_calls)."""
    calls = 0
    ids = list(range(len(docs)))
    if use_grader and docs:
        passages = '\n\n'.join(f'[{i}] {d["content"]}' for i, d in enumerate(docs))
        grader = call_llm(f'Select passages useful for this question. Return only JSON like {{"relevant_ids": [0]}}.\nQuestion: {query}\nPassages:\n{passages}')
        calls += 1
        try:
            ids = json.loads(re.search(r'\{.*\}', grader, re.DOTALL).group())['relevant_ids']
        except (AttributeError, json.JSONDecodeError, KeyError, TypeError):
            pass
    context = '\n\n---\n\n'.join(f'Source {i + 1}: {docs[i]["content"]}' for i in ids if isinstance(i, int) and 0 <= i < len(docs))
    if not context:
        return ABSTAIN, calls
    prompt = f"Answer only from this context. If absent, say: {ABSTAIN} Cite [Source n].\n\n{context}\n\nQuestion: {query}\nAnswer:"
    return call_llm(prompt), calls + 1

# ---- 5. Scoring
ABSTAIN_PATTERNS = r"i don'?t know|i do not know|not (?:mentioned|stated|provided|specified|given|included|available|found|contain)|no (?:information|mention|relevant)|does not (?:mention|state|provide|specify|contain|include)|doesn'?t (?:mention|say|state|provide)|cannot (?:be )?(?:determine|answer|find)|unable to"

def _norm(text):
    text = text.lower().replace('’', "'").replace('‑', '-').replace('–', '-').replace('—', '-').replace('*', '')
    return re.sub(r'\s+', '', text)

def keyword_score(answer, keywords):
    if not keywords:
        return None
    a = _norm(answer)
    return sum(any(_norm(alt) in a for alt in kw.split('|')) for kw in keywords) / len(keywords)

def is_abstention(answer):
    return bool(re.search(ABSTAIN_PATTERNS, answer.lower().replace('’', "'")))

def llm_judge(q, answer):
    """Optional, stricter but costs one extra LLM call per question."""
    verdict = call_llm(f"You grade answers. Question: {q['question']}\nReference answer: {q['reference']}\n"
                       f"Candidate answer: {answer}\nIs the candidate correct and consistent with the reference? "
                       f"Reply with exactly one word: CORRECT, PARTIAL or WRONG.")
    v = verdict.strip().upper()
    return 1.0 if v.startswith('CORRECT') else 0.5 if v.startswith('PARTIAL') else 0.0

def _qnorm(text):
    for a, b in [('’', "'"), ('‘', "'"), ('“', '"'), ('”', '"'), ('–', '-'), ('—', '-'), ('‑', '-')]:
        text = text.replace(a, b)
    return re.sub(r'\s+', '', text.lower())

def gold_labels(q):
    """Section IDs (AIcorpus set) or evidence indexes (legal set)."""
    return set(q['gold_sections']) if 'gold_sections' in q else set(range(len(q.get('evidence', []))))

def matched_labels(doc, q):
    if 'gold_sections' in q:
        return gold_labels(q) & set(doc['sections'])
    text = _qnorm(doc['content'])
    return {i for i, e in enumerate(q['evidence'])
            if doc['metadata'].get('source_file') == e['file'] and _qnorm(e['quote']) in text}

def retrieval_metrics(docs, q, k):
    gold = gold_labels(q)
    top = docs[:k]
    hits = [matched_labels(d, q) for d in top]
    rel = [bool(h) for h in hits]
    found = set().union(*hits) if hits else set()
    first = next((r + 1 for r, ok in enumerate(rel) if ok), None)
    return {'hit': float(any(rel)), 'section_recall': len(found) / len(gold),
            'precision': sum(rel) / max(len(top), 1), 'mrr': 1 / first if first else 0.0}

# ---- 6. Runner. `transform` turns a question into search queries (used by the query-rewriting cell below)
def no_rewrite(query):
    return {'dense': [query], 'sparse': [query], 'llm_calls': 0}

def run_eval(name='baseline', transform=no_rewrite, questions=None, generate=True, use_grader=True,
             final_k=5, use_llm_judge=False, adaptive_threshold=None, verbose=False):
    """adaptive_threshold: if set, first retrieve WITHOUT rewriting and only call `transform`
    when the best cross-encoder score is below the threshold (pays the rewrite cost only on hard queries)."""
    questions = questions or EVAL_QUESTIONS
    rows = []
    for n, q in enumerate(questions, 1):
        t0 = time.perf_counter()
        rewrite_calls, rewritten = 0, None
        if adaptive_threshold is not None:
            docs = hybrid_retrieve(q['question'], final_k=final_k)
            if not docs or docs[0]['rerank_score'] < adaptive_threshold:
                tq = transform(q['question'])
                rewrite_calls, rewritten = tq['llm_calls'], tq
                docs = hybrid_retrieve(q['question'], tq['dense'], tq['sparse'], final_k=final_k)
        else:
            tq = transform(q['question'])
            rewrite_calls, rewritten = tq['llm_calls'], (tq if tq['llm_calls'] else None)
            docs = hybrid_retrieve(q['question'], tq['dense'], tq['sparse'], final_k=final_k)
        t1 = time.perf_counter()
        answer, gen_calls = generate_answer(q['question'], docs, use_grader) if generate else ('', 0)
        t2 = time.perf_counter()

        row = {'id': q['id'], 'type': q['type'], 'question': q['question'],
               'gold': q.get('gold_sections') or sorted({e['file'] for e in q.get('evidence', [])}),
               'retrieved_sections': [d['sections'] for d in docs], 'top_rerank_score': docs[0]['rerank_score'] if docs else None,
               'rewritten': rewritten, 'answer': answer, 'retrieval_s': t1 - t0, 'generation_s': t2 - t1,
               'total_s': t2 - t0, 'llm_calls': rewrite_calls + gen_calls}
        if gold_labels(q):
            row.update(retrieval_metrics(docs, q, final_k))
        if generate:
            if q['type'] == 'unanswerable':
                row['abstained'] = is_abstention(answer)
                row['answer_score'] = float(row['abstained'])
            else:
                row['answer_score'] = keyword_score(answer, q['keywords'])
                row['abstained'] = row['answer_score'] == 0 and is_abstention(answer)   # refused a question it could answer
            if use_llm_judge and q['type'] != 'unanswerable':
                row['judge_score'] = llm_judge(q, answer)
        rows.append(row)
        if verbose or n % 25 == 0 or n == len(questions):
            print(f'  [{name}] {n}/{len(questions)}  avg {np.mean([r["total_s"] for r in rows]):.2f}s/q')

    (RESULTS_DIR / f'{name}.json').write_text(json.dumps(rows, indent=1, ensure_ascii=False), encoding='utf-8')
    return rows

def summarize(rows, name='', by_type=True):
    def avg(key, subset):
        vals = [r[key] for r in subset if r.get(key) is not None]
        return np.mean(vals) if vals else float('nan')
    def line(label, subset):
        unans = [r for r in subset if r['type'] == 'unanswerable']
        ans = [r for r in subset if r['type'] != 'unanswerable']
        return (f"{label:<16}{len(subset):>4}  hit@k {avg('hit', subset):.3f}  sec-recall {avg('section_recall', subset):.3f}  "
                f"MRR {avg('mrr', subset):.3f}  P@k {avg('precision', subset):.3f}  ans {avg('answer_score', ans):.3f}  "
                f"abstain-ok {avg('answer_score', unans):.3f}  false-abstain {avg('abstained', ans):.3f}  "
                f"{avg('total_s', subset):.2f}s/q  {avg('llm_calls', subset):.1f} calls/q")
    print(f'=== {name} ===')
    print(line('ALL', rows))
    if by_type:
        for t in sorted({r['type'] for r in rows}):
            print(line(t, [r for r in rows if r['type'] == t]))
    sm = lambda k: avg(k, [r for r in rows if r['type'] != 'unanswerable'])
    return {'name': name, 'hit': avg('hit', rows), 'section_recall': avg('section_recall', rows), 'mrr': avg('mrr', rows),
            'answer_score': sm('answer_score'), 'abstain_ok': avg('answer_score', [r for r in rows if r['type'] == 'unanswerable']),
            'judge_score': sm('judge_score'), 'avg_s': avg('total_s', rows),
            'p95_s': float(np.percentile([r['total_s'] for r in rows], 95)), 'llm_calls': avg('llm_calls', rows)}

def show_failures(rows, n=10):
    bad = [r for r in rows if r.get('hit') == 0.0 or (r.get('answer_score') is not None and r['answer_score'] < 0.5)]
    print(f'{len(bad)} questions with a retrieval miss or low answer score; first {n}:')
    for r in bad[:n]:
        print(f"- {r['id']} [{r['type']}] {r['question']}\n    gold {r['gold']}  got {r['retrieved_sections']}\n    answer: {r['answer'][:200]!r}")

In [ ]:
# Baseline run: no query rewriting. Set generate=False for a fast retrieval-only run that makes no Groq calls
baseline_rows = run_eval('baseline', generate=True)
baseline_summary = summarize(baseline_rows, 'baseline (no rewrite)')
show_failures(baseline_rows)

Query rewriting: time vs. accuracy trade-off (none / rewrite / multi-query / HyDE / adaptive). Writes eval_results/metrics_summary.md with resume-ready numbers.

In [ ]:
# ---------------- Query rewriting + time/accuracy trade-off ----------------
# Each strategy costs extra LLM calls (time) and may buy better retrieval (accuracy).
#   none        : original question                                  0 extra calls
#   rewrite     : LLM rewrites the question into a clean search query  1 extra call
#   multi_query : LLM writes 3 alternative queries, results fused (RRF) 1 extra call, 4x retrieval
#   hyde        : LLM writes a hypothetical answer passage; it is embedded for dense search  1 extra call
#   adaptive_*  : retrieve with the original first; rewrite ONLY if the best rerank score is low
# The cross-encoder always reranks against the ORIGINAL question, and the answer is generated
# for the original question - rewriting only changes what we search with.

REWRITE_PROMPT = """Rewrite the user's question into a single search query for retrieving passages from {corpus}.
Fix spelling, expand slang and abbreviations, name the exact technical concepts, people or systems involved, and keep it under 25 words.
Return only the query.
Question: {q}
Search query:"""

MULTI_QUERY_PROMPT = """Write 3 different search queries that would retrieve passages answering the question below from {corpus}.
Use different wording and technical terms in each. If the question asks about several things, give each its own query.
Return exactly 3 lines, one query per line, no numbering.
Question: {q}"""

HYDE_PROMPT = """Write a short, factual paragraph (3-4 sentences), in the style of {corpus}, that answers the question. Use precise technical or legal terms.
Question: {q}
Paragraph:"""

def _clean(text):
    return text.strip().strip('"').strip()

def rewrite_query(q):
    rq = _clean(call_llm(REWRITE_PROMPT.format(q=q, corpus=CORPUS_DESCRIPTION))) or q
    return {'dense': [rq], 'sparse': [q + ' ' + rq], 'llm_calls': 1, 'text': rq}

def multi_query(q):
    lines = [re.sub(r'^\s*(?:\d+[.)]|[-*•])\s*', '', l).strip() for l in call_llm(MULTI_QUERY_PROMPT.format(q=q, corpus=CORPUS_DESCRIPTION)).splitlines()]
    variants = [l for l in lines if l][:3]
    return {'dense': [q] + variants, 'sparse': [q] + variants, 'llm_calls': 1, 'text': variants}

def hyde(q):
    passage = _clean(call_llm(HYDE_PROMPT.format(q=q, corpus=CORPUS_DESCRIPTION))) or q
    return {'dense': [q, passage], 'sparse': [q], 'llm_calls': 1, 'text': passage}

QUERY_TRANSFORMS = {'none': no_rewrite, 'rewrite': rewrite_query, 'multi_query': multi_query, 'hyde': hyde}

def rag_with_rewrite(query, mode='rewrite', final_k=5, use_grader=True, adaptive_threshold=None, show=True):
    """Interactive use: ask one question with a chosen rewriting strategy and see the cost."""
    t0 = time.perf_counter()
    transform = QUERY_TRANSFORMS[mode]
    tq = no_rewrite(query)
    docs = hybrid_retrieve(query, final_k=final_k) if adaptive_threshold is not None else None
    if docs is None or not docs or docs[0]['rerank_score'] < adaptive_threshold:
        tq = transform(query)
        docs = hybrid_retrieve(query, tq['dense'], tq['sparse'], final_k=final_k)
    t1 = time.perf_counter()
    answer, calls = generate_answer(query, docs, use_grader)
    t2 = time.perf_counter()
    if show:
        print(f'mode={mode}  search with: {tq.get("text", query)}')
        print(f'sections: {[d["sections"] for d in docs]}')
        print(f'retrieval {t1 - t0:.2f}s, generation {t2 - t1:.2f}s, LLM calls {tq["llm_calls"] + calls}\n')
        print(answer)
    return answer

# ---------------- Run the comparison ----------------
# Groq free-tier rate limits make 200 questions x 5 strategies x 2-3 LLM calls slow.
# Knobs:  GENERATE=False -> retrieval-only comparison (only the rewrite call hits the LLM; fast)
#         N_QUESTIONS    -> evaluate a stratified sample instead of all ~200
GENERATE = True
USE_GRADER = True          # False = one fewer LLM call per question (faster, slightly less precise context)
USE_LLM_JUDGE = False      # True = extra LLM call per question for a stricter answer score
N_QUESTIONS = None         # e.g. 50 for a quick run
ADAPTIVE_THRESHOLD = 0.0   # cross-encoder logit; below this the question is considered "hard" and gets rewritten

def stratified_sample(questions, n):
    if not n or n >= len(questions):
        return questions
    by_type = {}
    for q in questions:
        by_type.setdefault(q['type'], []).append(q)
    picked = []
    for t, qs in by_type.items():
        k = max(1, round(n * len(qs) / len(questions)))
        picked += qs[:: max(1, len(qs) // k)][:k]
    return picked

eval_set = stratified_sample(EVAL_QUESTIONS, N_QUESTIONS)
strategies = [
    ('none',                 no_rewrite,     None),
    ('rewrite',              rewrite_query,  None),
    ('multi_query',          multi_query,    None),
    ('hyde',                 hyde,           None),
    ('adaptive_multi_query', multi_query,    ADAPTIVE_THRESHOLD),
]

tradeoff, all_rows = [], {}
for name, transform, threshold in strategies:
    rows = run_eval(name, transform, eval_set, generate=GENERATE, use_grader=USE_GRADER,
                    use_llm_judge=USE_LLM_JUDGE, adaptive_threshold=threshold)
    all_rows[name] = rows
    tradeoff.append(summarize(rows, name, by_type=False))
    if threshold is not None:
        print(f"  rewrote {sum(r['rewritten'] is not None for r in rows)}/{len(rows)} questions")

print('\n================ TIME vs ACCURACY ================')
base = tradeoff[0]
acc_key = 'judge_score' if USE_LLM_JUDGE else ('answer_score' if GENERATE else 'section_recall')
print(f"{'strategy':<22}{'hit@5':>7}{'recall':>8}{'MRR':>7}{'answer':>8}{'abstain':>9}{'avg s':>8}{'p95 s':>8}{'calls':>7}{'Δacc':>8}{'Δtime':>8}")
for s in tradeoff:
    print(f"{s['name']:<22}{s['hit']:>7.3f}{s['section_recall']:>8.3f}{s['mrr']:>7.3f}{s['answer_score']:>8.3f}{s['abstain_ok']:>9.3f}"
          f"{s['avg_s']:>8.2f}{s['p95_s']:>8.2f}{s['llm_calls']:>7.1f}{s[acc_key] - base[acc_key]:>+8.3f}{s['avg_s'] / base['avg_s']:>7.1f}x")

# per-type view: rewriting usually helps the vague 'conversational' and 'multi' questions most
for t in sorted({q['type'] for q in eval_set}):
    cells = []
    for name, rows in all_rows.items():
        sub = [r for r in rows if r['type'] == t]
        key = 'answer_score' if GENERATE else 'section_recall'
        vals = [r[key] for r in sub if r.get(key) is not None]
        cells.append(f"{name}={np.mean(vals):.2f}" if vals else f"{name}=n/a")
    print(f"{t:<15}", '  '.join(cells))

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for s in tradeoff:
        ax.scatter(s['avg_s'], s[acc_key], s=60)
        ax.annotate(s['name'], (s['avg_s'], s[acc_key]), textcoords='offset points', xytext=(6, 4))
    ax.set_xlabel('average seconds per question')
    ax.set_ylabel(acc_key)
    ax.set_title('Query rewriting: time vs accuracy')
    ax.grid(alpha=0.3)
    plt.show()
except ImportError:
    print('(pip install matplotlib to see the trade-off plot)')

# Try one yourself:
# rag_with_rewrite("is software patentable in india", mode='multi_query')

# ---------------- Resume-ready summary (numbers come from YOUR run above) ----------------
best = max(tradeoff, key=lambda s: s[acc_key])
fast = tradeoff[0]
n_q = len(eval_set)
pct = lambda x: f'{100 * x:.1f}%'
lines = [
    f'# RAG evaluation summary ({QUESTIONS_FILE}: {n_q} questions, {len(eval_docs)} chunks)',
    '',
    '| strategy | Hit@5 | Section recall@5 | MRR | Answer score | Abstention acc. | Avg latency | LLM calls/q |',
    '|---|---|---|---|---|---|---|---|',
] + [f"| {s['name']} | {pct(s['hit'])} | {pct(s['section_recall'])} | {s['mrr']:.3f} | "
     f"{pct(s['answer_score']) if GENERATE else 'n/a'} | {pct(s['abstain_ok']) if GENERATE else 'n/a'} | "
     f"{s['avg_s']:.2f}s | {s['llm_calls']:.1f} |" for s in tradeoff] + [
    '',
    '## Resume bullets (edit to taste)',
    f"- Built a hybrid RAG pipeline (MiniLM dense + BM25, reciprocal rank fusion, cross-encoder reranking, LLM passage grading) "
    f"and evaluated it on a {n_q}-question labelled benchmark (single-hop, multi-hop, distractor, conversational and unanswerable).",
    f"- Reached {pct(best['hit'])} Hit@5 and {best['mrr']:.2f} MRR"
    + (f", with {pct(best['abstain_ok'])} correct abstention on unanswerable questions" if GENERATE else '') + '.',
    f"- Added query rewriting (LLM rewrite, multi-query, HyDE, adaptive); best strategy '{best['name']}' changed {acc_key} by "
    f"{100 * (best[acc_key] - fast[acc_key]):+.1f} points at {best['avg_s'] / fast['avg_s']:.1f}x latency vs. no rewriting.",
]
summary_md = '\n'.join(lines)
(RESULTS_DIR / 'metrics_summary.md').write_text(summary_md, encoding='utf-8')
print(summary_md)
print(f'\nsaved to {RESULTS_DIR / "metrics_summary.md"}')